In [1]:
import os
import re
import logging

from dotenv import load_dotenv
from youtube_transcript_api import YouTubeTranscriptApi, TranscriptsDisabled, NoTranscriptFound

In [2]:
load_dotenv()

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s %(levelname)s %(message)s",
)
logger = logging.getLogger(__name__)

In [ ]:
def extract_video_id(url: str) -> str:
    """Extract and return the video ID from a YouTube URL."""
    patterns = [
        r"(?:v=)([A-Za-z0-9_-]{11})",          # ?v=ID
        r"youtu\.be/([A-Za-z0-9_-]{11})",       # youtu.be/ID
        r"shorts/([A-Za-z0-9_-]{11})",           # /shorts/ID
    ]
    for pattern in patterns:
        match = re.search(pattern, url)
        if match:
            return match.group(1)
    raise ValueError(f"Could not extract a YouTube video ID from: {url}")

In [6]:
def get_transcript(url: str) -> str:
    """Fetch and return the full transcript for a YouTube video."""
    video_id = extract_video_id(url)
    logger.info("Fetching transcript for video_id=%s", video_id)

    try:
        api = YouTubeTranscriptApi()
        fetched = api.fetch(video_id)
    except TranscriptsDisabled:
        raise RuntimeError(f"Transcripts are disabled for video: {video_id}")
    except NoTranscriptFound:
        raise RuntimeError(f"No transcript found for video: {video_id}")
    except Exception as exc:
        raise RuntimeError(
            f"Failed to fetch transcript for video: {video_id}"
        ) from exc

    transcript = " ".join(snippet.text for snippet in fetched)
    logger.info("Transcript fetched: %d characters", len(transcript))
    return transcript

In [7]:
TEST_URL = "https://www.youtube.com/watch?v=dQw4w9WgXcQ"

video_id = extract_video_id(TEST_URL)
print("Video ID:", video_id)

transcript = get_transcript(TEST_URL)
print("Transcript length:", len(transcript))

2026-05-18 21:44:47,586 INFO Fetching transcript for video_id=dQw4w9WgXcQ


Video ID: dQw4w9WgXcQ


2026-05-18 21:44:49,068 INFO Transcript fetched: 2089 characters


Transcript length: 2089


In [8]:
word_count = len(transcript.split())
print("Word count:", word_count)
print("First 200 chars:", transcript[:200])
print("Last 200 chars: ", transcript[-200:])

Word count: 487
First 200 chars: [♪♪♪] ♪ We're no strangers to love ♪ ♪ You know the rules
and so do I ♪ ♪ A full commitment's
what I'm thinking of ♪ ♪ You wouldn't get this
from any other guy ♪ ♪ I just wanna tell you
how I'm feelin
Last 200 chars:  ou ♪ ♪ Never gonna give you up ♪ ♪ Never gonna let you down ♪ ♪ Never gonna run around
and desert you ♪ ♪ Never gonna make you cry ♪ ♪ Never gonna say goodbye ♪ ♪ Never gonna tell a lie
and hurt you ♪
